In [2]:
import math 
import time
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import os
import torch
import cv2
import torchvision.transforms as transforms
from torch.utils.data import Dataset , DataLoader, random_split
from dotenv import load_dotenv
import numpy as np
import torch.nn as nn
import torch.optim as optim
from imgDataset import imgDataset
from EAST import EAST

imported imgDataset.py
torch version: 2.9.1+cu128


In [3]:
# loss funtions
# ls
def balanced_cross_entropy_loss(preds, targets, epsilon=1e-7) -> torch.Tensor:
    beta = 1 - torch.mean(targets.float())
    preds = torch.clamp(preds, epsilon, 1.0 - epsilon)
    return ((-beta * targets * torch.log(preds)) - 
            (1-beta)*(1-targets)*(torch.log(1-preds))).mean()

#lg this is what is breaking
def quad_loss(preds, targets, score_map, epsilon=1e-7) -> torch.Tensor :
    mask = score_map  # [B, 1, H, W]
    loss = torch.nn.functional.smooth_l1_loss(preds * mask, targets * mask, reduction='sum')
    normalizer =(mask.sum() + epsilon) * 8
    return loss / ( normalizer)


def get_iou(pred: torch.Tensor, gt: torch.Tensor, smooth: float = 1e-6) -> float:
    intersection = torch.logical_and(gt, pred).sum().float()
    union = torch.logical_or(gt, pred).sum().float()
    iou = (intersection + smooth) / (union + smooth)
    return iou.item()

In [4]:
def train_cycle(model, dataset_loaded, device, optimizer, criterion, train=True, width=640, height=360):
    correct = 0
    total = 0
    start = time.time()
    running_loss = 0.0
    count = 0
    for imgs, _, coords, corners in dataset_loaded:

        count += len(coords)
        imgs = imgs.to(device)
        coords = coords.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()
        pred_score_map, pred_geo_map = model(imgs)

        loss_score_map = balanced_cross_entropy_loss(pred_score_map.squeeze(1), coords.squeeze(1))
        total_loss = loss_score_map 
        
        #loss_score_map = balanced_cross_entropy_loss(pred_score_map.squeeze(1), coords)
        #loss_geo_map = quad_loss(pred_geo_map, corners, coords)
        #total_loss = loss_score_map + (1.0 * loss_geo_map)
        

        total_loss.backward()
        optimizer.step()
        running_loss += total_loss.item()

    avg_loss = running_loss / len(dataset_loaded)
    print(f"Time taken: {time.time() - start:.2f} seconds")
    print(f"Avg Loss: {avg_loss:.4f}")
    return avg_loss

def val_cycle(model, dataset_loaded, device, optimizer, criterion, train=True, width=640, height=360):
    model.eval()
    start = time.time()
    running_loss = 0.0
    ious = 0.0
    
    with torch.no_grad():
        for imgs, _, coords, corners in dataset_loaded:

            imgs = imgs.to(device)
            coords = coords.to(device)
            corners = corners.to(device)

            pred_score_map, pred_geo_map = model(imgs)
            print(pred_score_map.min(), pred_score_map.max(), pred_score_map.mean())


            loss_score_map = balanced_cross_entropy_loss(pred_score_map.squeeze(1), coords.squeeze(1))
            
            #loss_geo_map = quad_loss(pred_geo_map, corners, coords)
            #total_loss = loss_score_map + (1.0 * loss_geo_map)
            total_loss = loss_score_map

            running_loss += total_loss.item()
            threshold = pred_score_map.mean().item()

            predicted = (pred_score_map.squeeze(1) > threshold).float()
            gt = coords.squeeze(1)
            ious += get_iou(predicted, gt)

    
    avg_loss = running_loss / len(dataset_loaded)
    iou = ious / len(dataset_loaded)

    print(f"Time taken: {time.time() - start:.2f} seconds")
    print(f"Avg Loss: {avg_loss:.4f}, IoU: {iou:.4f}")
    return avg_loss, iou
print("ew...")

ew...


In [5]:
def train_model(model, loader_train, loader_val, 
                criterion,scheduler, optimizer, cycles, 
                width = 640, height = 360):
    model.train()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)

    # best acc and where to store the model
    best_val_acc = 0.0
    best_model_path = "east_model.pth"

    for cycle in range(cycles):
        print(f"cycle: {cycle+1}/{cycles}")

        #training within the cycle
        model.train()
        train_loss = train_cycle(model, dataset_loaded = loader_train, 
                                            device = device,optimizer = optimizer, 
                                            criterion = criterion, train = True, width = width,
                                            height = height)
        
        #to test with the cycle after traing 
        model.eval()
        val_loss, val_acc = val_cycle(model, dataset_loaded = loader_val, 
                                            device = device,optimizer = optimizer, 
                                            criterion = criterion, train = False, width = width,
                                            height = height)
        scheduler.step()
        
        #to compare and see if there was inprovement
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f"model beats old with Val Acc: {best_val_acc:.2f}%, saving model.")
            torch.save(model.state_dict(), "east_model.pth")

    return best_model_path

    



In [6]:
#loads in paths
load_dotenv()
directory_train = os.getenv('Directory_train')
directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')
width = 640
height = 360

In [7]:
# Define path_imgs and path_labels
path_imgs = sorted(os.listdir(directory_train))
path_labels = sorted(os.listdir(directory_train_textAndCoords))

# Generate paths for training images and labels
train_img_paths = [os.path.join(directory_train, f) for f in path_imgs]
train_label_paths = [os.path.join(directory_train_textAndCoords, f) for f in path_labels]

In [8]:
#making objs
train_dataset = [train_img_paths, train_label_paths]
train_dataset = imgDataset(train_img_paths, train_label_paths, img_width= width, img_height= height)
img, labels, coords, corners = train_dataset[0]

print(f"Image shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
print(f"First Coords: {coords.shape}",type(coords))

#batch = custom_collate(train_dataset)


Image shape: torch.Size([1, 360, 640]) <class 'torch.Tensor'>
First Coords: torch.Size([1, 360, 640]) <class 'torch.Tensor'>


In [ ]:
#spltining the dataset into training and validation sets
val_split = 0.2
training_size = int((1 - val_split) * len(train_dataset))
val_size = len(train_dataset) - training_size
training_dataset, val_dataset = random_split(train_dataset, [training_size, val_size])
loader_train = DataLoader(training_dataset, batch_size=4, shuffle=True,num_workers=16, pin_memory=True)
loader_val = DataLoader(val_dataset, batch_size=4, shuffle=True,num_workers=8, pin_memory=True)
images, labels = next(iter(loader_train))
print("=== Training Batch Shapes ===")
print(f"Images Tensor Shape (N, C, H, W): {images.shape}")
print(f"Labels Tensor Shape:              {labels.shape}")



Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f64133fff60>
Traceback (most recent call last):
  File "/home/dan/Documents/projects/EAST_model_Pytorch/model/venv/lib64/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/home/dan/Documents/projects/EAST_model_Pytorch/model/venv/lib64/python3.12/site-packages/torch/utils/data/dataloader.py", line 1618, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib64/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/u

=== Training Batch Shapes ===


NameError: name 'images' is not defined

In [ ]:
#loading model
model = EAST(color_channel=1, scale_factor=4, img_height = height , img_width = width)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.Initialize_weights()
model.to(device)


In [ ]:
# Loading model if it exists
load_path = os.getenv('Load_model')

if load_path and os.path.isfile(load_path):
    model.load_state_dict(torch.load(load_path))
    print(f"Model loaded successfully from: {load_path}")
else:
    print("Model not loaded, starting from scratch")

In [ ]:
#defualt model otipions
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
cycles = 10

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
train_model(model = model, loader_train = loader_train, loader_val = loader_val, criterion = criterion,scheduler=scheduler, optimizer = optimizer, cycles = cycles, width = width, height = height)

In [ ]:
model_save_path = os.getenv('Model_save_path', 'east_model.pth')
torch.save(model.state_dict(), model_save_path)